In [0]:
%sql
create schema if not exists databricks_catalog.gold

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
init_load_flag = dbutils.widgets.get("init_load_flag")

# Data Reading

In [0]:
df=spark.sql("Select * from databricks_catalog.silver.silver_customers")
df.display()

customer_id,email,city,state,domain,Fullname
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers


# Removing Duplicates

In [0]:
df = df.dropDuplicates(subset=['customer_id'])

# Dividing new vs old records

In [0]:
if init_load_flag == 0:
    df_old = spark.sql("""
        select DimCustomerKey, customer_id, create_date, update_date
        from databricks_catalog.gold.DimCustomers
    """)
else:
    df_old = spark.sql("""
        select 0 DimCustomerKey, 0 customer_id, 0 create_date, 0 update_date
        from databricks_catalog.silver.silver_customers where 1=0
    """)

In [0]:
df_old.display()

DimCustomerKey,customer_id,create_date,update_date


# Renaming coloumns df_old

In [0]:
df_old = (df_old.withColumnRenamed("DimCustomerKey","oldDimCustomerKey")
                            .withColumnRenamed("customer_id","oldcustomer_id")
                            .withColumnRenamed("create_date","oldcreate_date")
                            .withColumnRenamed("update_date","oldupdate_date"))
df_old.display()



oldDimCustomerKey,oldcustomer_id,oldcreate_date,oldupdate_date


# Applying join with old records

In [0]:
df_join = df.join(
    df_old,
    df.customer_id == df_old.oldcustomer_id,
    "left"
)

# Seperating new vs old records

In [0]:
df_new = df_join.filter("oldDimCustomerKey IS NULL")

In [0]:
df_new.limit(10).display()

customer_id,email,city,state,domain,Fullname,oldDimCustomerKey,oldcustomer_id,oldcreate_date,oldupdate_date
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney,null,null,null,null
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers,null,null,null,null
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes,null,null,null,null
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott,null,null,null,null
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez,null,null,null,null
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,null,null,null,null
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle,null,null,null,null
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos,null,null,null,null
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green,null,null,null,null
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,null,null,null,null


In [0]:
df_old = df_join.filter("oldDimCustomerKey IS NOT NULL")

# Preparing df_old

In [0]:
# Dropping all the columns which are not required
df_old = df_old.drop('old_DimCustomerKey', 'old_customer_id', 'old_update_date')

# Renaming "old_create_date" column to "create_date"
df_old = df_old.withColumnRenamed("old_create_date", "create_date")

# Recreating "update_date" column with current timestamp
df_old = df_old.withColumn("update_date", current_timestamp())


# Preparing df_new

In [0]:
# Dropping all the columns which are not required
df_new = df_new.drop('old_DimCustomerKey', 'old_customer_id', 'old_update_date', 'old_create_date')

# Recreating "update_date", "current_date" columns with current timestamp
df_new = df_new.withColumn("update_date", current_timestamp())
df_new = df_new.withColumn("current_date", current_timestamp())

In [0]:
df_new.display()  # Ensure 'old_DimCustomerKey' exists in df_new before display

customer_id,email,city,state,domain,Fullname,oldDimCustomerKey,oldcustomer_id,oldcreate_date,oldupdate_date,update_date,current_date,DimCustomerKey
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,1
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,2
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,3
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,4
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,5
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,6
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,7
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,8
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,9
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,null,null,null,null,2026-08-27T03:23:08.093Z,2026-08-27T03:23:08.093Z,10


In [0]:
df_new.display()


customer_id,email,city,state,domain,Fullname,oldDimCustomerKey,oldcustomer_id,oldcreate_date,oldupdate_date,update_date,current_date,DimCustomerKey
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,1
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,2
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,3
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,4
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,5
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,6
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,7
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,8
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,9
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,null,null,null,null,2026-08-27T03:23:18.089Z,2026-08-27T03:23:18.089Z,10



# Surrogate key-all the values

In [0]:
df_new = df_new.withColumn(
    "Dimcustomerkey",
    monotonically_increasing_id() + lit(1)
)

df_new.limit(10).display()

customer_id,email,city,state,domain,Fullname,oldDimCustomerKey,oldcustomer_id,oldcreate_date,oldupdate_date,update_date,current_date,Dimcustomerkey
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,1
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,2
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,3
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,4
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,5
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,6
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,7
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,8
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,9
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,null,null,null,null,2026-08-27T03:23:23.390Z,2026-08-27T03:23:23.390Z,10


# Adding max surrogate key

In [0]:
init_load_flag = dbutils.widgets.get("init_load_flag")
print(init_load_flag)

init_load_flag


In [0]:
init_load_flag = "1"

if init_load_flag == "1":
    max_surrogate_key = 0
else:
    df_maxsur = spark.sql("""
        SELECT MAX(DimCustomerKey) AS max_surrogate_key
        FROM databricks_catalog.silver.DimCustomers
    """)



    max_surrogate_key = df_maxsur.collect()[0]["max_surrogate_key"]

print(max_surrogate_key)

0


In [0]:
df_new = df_new.withColumn(
    "DimCustomerKey",
    lit(max_surrogate_key) + col("DimCustomerKey")
)

# Union of df_old and df_new

In [0]:
df_final = df_new.unionByName(df_old, allowMissingColumns=True)

# SCD Type1

In [0]:
df_final.display()

customer_id,email,city,state,domain,Fullname,oldDimCustomerKey,oldcustomer_id,oldcreate_date,oldupdate_date,update_date,current_date,DimCustomerKey
C00001,rushjeff@ryan.org,Johnsonmouth,MS,ryan.org,Emily Mooney,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,1
C00002,mccoykiara@kelly.com,Stephenfort,WY,kelly.com,Andrea Sellers,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,2
C00003,rebeccamiller@yahoo.com,South Stephenshire,LA,yahoo.com,Craig Hayes,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,3
C00004,lawrence05@campbell.info,Chrisland,ND,campbell.info,Bryan Scott,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,4
C00005,carrie45@yahoo.com,East Dennistown,RI,yahoo.com,Sean Vasquez,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,5
C00006,traceyramos@gmail.com,North Matthew,IN,gmail.com,Kevin Mccarthy,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,6
C00007,scottallen@gmail.com,Joneshaven,VA,gmail.com,Amanda Doyle,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,7
C00008,sullivanjeremy@horton-adams.com,South Nathanfurt,CT,horton-adams.com,Paul Campos,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,8
C00009,dennis03@yahoo.com,Kimberlyview,MD,yahoo.com,Mary Green,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,9
C00010,charles58@murillo.net,West Hector,OK,murillo.net,James Myers,null,null,null,null,2026-08-27T03:27:34.554Z,2026-08-27T03:27:34.554Z,10


In [0]:
from delta.tables import DeltaTable

if not spark.catalog.tableExists("databricks_catalog.gold.DimCustomers"):

    df_final.write \
        .format("delta") \
        .mode("overwrite") \
        .option(
            "path",
            "abfss://gold@databricksetlpractice.dfs.core.windows.net/DimCustomers"
        ) \
        .saveAsTable("databricks_catalog.gold.DimCustomers")

else:

    dlt_obj = DeltaTable.forPath(
        spark,
        "abfss://gold@databricksetlpractice.dfs.core.windows.net/DimCustomers"
    )

    dlt_obj.alias("trg").merge(
        df_final.alias("src"),
        "trg.DimCustomerKey = src.DimCustomerKey"
    ) \
    .whenMatchedUpdateAll() \
    .whenNotMatchedInsertAll() \
    .execute()